# Text-Image In-Batch Mining — Remaining Sizes (N=20000, N=33208)

Continues the in-batch mining sweep for the two largest sizes, reading and writing to the SAME output
directory as the completed run (100 through 10000 already done and saved there). Epoch counts match the
tapered schedule used throughout this sweep (8 epochs at N=20000, 5 at N=33208), not a flat count.

**Time-optimization changes for these two large sizes:**
- Logging interval widened to every 500 steps (was 50).
- Checkpoint interval widened to every 500 steps (was 100).

**Hard/soft classification and counting is DELIBERATELY KEPT, unlike an earlier version of
this notebook.** The rate of anchors with zero hard negatives per batch is expected to rise as training
size grows — a fixed batch of 32 becomes a smaller, more scattered slice of a much larger, more diverse
pool as N increases, so the chance any two batch members happen to share season/region drops. This is
exactly the size range where that trend matters most to observe, and its actual impact on learning is not
yet established — so visibility into it is kept even at the cost of the extra per-batch classification
overhead.

Everything else — architecture, batch size 32, in-batch mining logic, masking rule, GT evaluation
protocol — is unchanged from the completed run.

## Section 0 installations

In [14]:
!pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable


In [17]:
!pip install openpyxl --upgrade

Defaulting to user installation because normal site-packages is not writeable


In [18]:
!pip install pandas --upgrade

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 46.4 MB/s  0:00:00m0:00:01


In [15]:
! pip install accelerate

Defaulting to user installation because normal site-packages is not writeable
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [15]:
!pip install transformers==4.57.6 #downgrade for DINOv3

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 19.7 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 6.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 18.1 MB/s  0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:━━━━━━━━ 0/3 [huggingface-hub]
      Successfully uninstalled huggingface_hub-1.29.00/3 [huggingface-hub]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/3 [huggingface-hub]  WARNING: The scripts hf, huggingface-cli and tiny-agents are installed in '/home/sjiang/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Attempting uninstall: tokenizers━━━━━━━━━━━━━━ 0/3 [huggingface-hub]
    Found existing installation: tokenizers 0.23.10/3 [huggingfa

In [31]:
!pip install --upgrade huggingface_hub

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 6.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 28.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 2/3 [huggingface_hub]  WARNING: The scripts hf, huggingface-cli and tiny-agents are installed in '/home/sjiang/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [huggingface_hub] [huggingface_hub]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spin 0.17 requires click!=8.3.0,<8.4,>=8, but you have click 8.5.0 which is incompatible.


## Section 1 — Setup, logging configuration

In [2]:
# import os
# os.environ['HF_HOME'] = '/projects/agrifutur/hf_cache2' # set global cache path for huggingface, not working

In [1]:
import json
import os
import re
import sys
import time
import math
import random
import logging
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from transformers import AutoModel, AutoImageProcessor, AutoTokenizer

import transformers

import openpyxl

In [ ]:
transformers.logging.set_verbosity_error()

LOG_FORMAT = '%(asctime)s : %(levelname)s : %(message)s'
logging.basicConfig(filename='/projects/agrifutur/sji/img_inbatch_mining_ft.log', level=logging.INFO, format=LOG_FORMAT)
logger = logging.getLogger("img_inbatch_ft")

device = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: %s", device)
if device == "cuda":
    logger.info("GPU: %s", torch.cuda.get_device_name(0))

import pickle
def pkl_vars(varname, filename):
    with open(filename, 'wb') as file:
        pickle.dump(varname, file)
        
def reload_vars(filename):
    this_var = None
    with open(filename, 'rb') as file:
        this_var = pickle.load(file)      
    return this_var

## Section 2 — Build bulletin -> season/region mapping
Config
Same `OUTPUT_DIR` as the completed run, so this reads existing progress and writes new sizes alongside it.

In [3]:
HF_CACHE = "/projects/agrifutur/hf_cache2"
BASE_DIR = "/projects/agrifutur/yabusaqe/BSV Sample Construction"
KG_EXPORT_PATH = os.path.join(BASE_DIR, "URLS_Meta.txt")
TEXTIMAGE_PAIRS_PATH = os.path.join(BASE_DIR, "Automatic Construction", "bsv_pairs_auto_textimage.jsonl")
IMAGE_DIR = os.path.join(BASE_DIR, "Automatic Construction", "bsv_images_auto_textimage")

LEAK_FLAGS_PATH = "/projects/agrifutur/yabusaqe/Finetuning/Shared_data/table_leak_flags_COMPLETE.xlsx"

DINOV3_MODEL_NAME = "facebook/dinov3-vitb16-pretrain-lvd1689m"
BGE_MODEL_NAME = "BAAI/bge-m3"
BGE_MAX_LENGTH = 512

N_UNFROZEN_LAYERS_DINOV3 = 6
N_UNFROZEN_LAYERS_BGE = 6

SHARED_DIM = 512
BATCH_SIZE = 32
HEAD_LEARNING_RATE = 1e-4
DINOV3_ENCODER_LR = 1e-6
BGE_ENCODER_LR = 1e-5
TEMPERATURE = 0.07

GT_TEXTIMAGE_PATH = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_gt_textimage_final_500.jsonl"
GT_IMAGE_DIR = "/projects/agrifutur/yabusaqe/BSV Sample Construction/Ground Truth/bsv_images_gt_2404"
K_VALUES = [1, 3, 5, 10]

RANDOM_SEED = 42
MAX_RUNTIME_HOURS_PER_SIZE = 7.5
IMAGE_ENCODE_CHUNK_SIZE = 16

# --- Only the two remaining large sizes ---
REMAINING_SIZES = [ 2500, 5000, 10000]#[100,500, 1000, 20000, 33208]

# --- Big-size time-optimization settings (this notebook's entire scope) ---
LOG_EVERY_N_STEPS = 500       # was 50
CHECKPOINT_EVERY_N_STEPS = 500  # was 100

# --- SAME output directory as the completed run -- reads existing progress, writes new sizes alongside it ---
OUTPUT_DIR = "/projects/agrifutur/sji/cross_encoder_ft/mined_negatives_ft/textimage_inbatch_mining_sweep"

logger.info("Config loaded.")
logger.info("OUTPUT_DIR exists: %s", os.path.exists(OUTPUT_DIR))
logger.info("TEXTIMAGE_PAIRS_PATH exists: %s", os.path.exists(TEXTIMAGE_PAIRS_PATH))
logger.info("LEAK_FLAGS_PATH exists: %s", os.path.exists(LEAK_FLAGS_PATH))
logger.info("GT_IMAGE_DIR exists: %s", os.path.exists(GT_IMAGE_DIR))

In [4]:
bulletin_to_meta_raw = reload_vars("vars/bulletin_to_meta_raw.pkl")
bulletin_meta = reload_vars("vars/bulletin_meta.pkl")
#run next cell if not exist

In [13]:
# def parse_kg_export(path):
#     with open(path, encoding="utf-8") as f:
#         lines = f.read().split("\n")

#     header_idx = None
#     for i, l in enumerate(lines):
#         if l.strip().rstrip("\r") == "bulletin\tdate\tregion":
#             header_idx = i
#             break
#     if header_idx is None:
#         raise ValueError(f"Could not find header row in {path}")

#     row_pattern = re.compile(
#         r'^bsv:bulletin_(?P<qid>Q\d+)_(?P<kg_year>\d{4})_(?P<rest>.+?) \[https\]\t"(?P<date>[\d-]+)"\twde:(?P<region_qid>Q\d+) \[http\]$'
#     )

#     bulletin_to_meta = {}
#     for l in lines[header_idx + 1:]:
#         l = l.rstrip("\r").strip()
#         if not l or l == "Powered by INRAE":
#             continue
#         m = row_pattern.match(l)
#         if not m:
#             continue
#         key = m.group("rest")
#         if key not in bulletin_to_meta:
#             bulletin_to_meta[key] = {"date": m.group("date"), "region_qid": m.group("region_qid")}
#     return bulletin_to_meta


# def month_to_season(month):
#     if month in (3, 4, 5):
#         return "printemps"
#     elif month in (6, 7, 8):
#         return "ete"
#     elif month in (9, 10, 11):
#         return "automne"
#     else:
#         return "hiver"


# bulletin_to_meta_raw = parse_kg_export(KG_EXPORT_PATH)
# logger.info("Parsed %d bulletin -> {date, region_qid} mappings", len(bulletin_to_meta_raw))
# pkl_vars(bulletin_to_meta_raw, "vars/bulletin_to_meta_raw.pkl")

# bulletin_meta = {}
# for bid, meta in bulletin_to_meta_raw.items():
#     try:
#         dt = datetime.strptime(meta["date"], "%Y-%m-%d")
#     except ValueError:
#         continue
#     bulletin_meta[bid] = {"season": month_to_season(dt.month), "region_qid": meta["region_qid"]}
# logger.info("Enriched with season: %d bulletins", len(bulletin_meta))
# pkl_vars(bulletin_meta, "vars/bulletin_meta.pkl")

### dinov3 is a gated model. need to 1) ask for grant via huggingface 2) login for first dowload

In [ ]:
#from huggingface_hub import login
#login()

In [5]:
from transformers.image_utils import load_image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = load_image(url)

pretrained_model_name = "facebook/dinov3-convnext-tiny-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(pretrained_model_name, cache_dir=HF_CACHE)
model = AutoModel.from_pretrained(
    pretrained_model_name, 
    device_map="auto", 
    cache_dir=HF_CACHE
)

## Section 3 — Load data, leak-filter

In [6]:
raw_textimage = reload_vars("vars/raw_textimage.pkl")
textimage_records = reload_vars("vars/textimage_records.pkl")
FULL_POOL_SIZE = reload_vars("vars/FULL_POOL_SIZE.pkl")
#run next cell if not exist

In [ ]:
# def load_jsonl(path):
#     rows = []
#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if line:
#                 rows.append(json.loads(line))
#     return rows

# raw_textimage = load_jsonl(TEXTIMAGE_PAIRS_PATH)
# logger.info("Text/image pairs: %d rows", len(raw_textimage))
# pkl_vars(raw_textimage, "vars/raw_textimage.pkl")

# textimage_records = []
# for i, r in enumerate(raw_textimage):
#     if r.get("pair_type") != "text_image":
#         continue
#     text = r.get("text", "").strip()
#     filename = r.get("image", {}).get("filename")
#     if not text or not filename:
#         continue
#     bid = r.get("bulletin_id")
#     if bid not in bulletin_meta:
#         continue
#     img_path = os.path.join(IMAGE_DIR, filename)
#     textimage_records.append({
#         "orig_idx": i, "bulletin_id": bid, "text": text, "image_path": img_path,
#         "season": bulletin_meta[bid]["season"], "region_qid": bulletin_meta[bid]["region_qid"],
#     })

# logger.info("Usable text/image records (with KG metadata): %d", len(textimage_records))


# leak_df = pd.read_excel(LEAK_FLAGS_PATH)
# leaked_indices = set(leak_df["row_index"])

# before_count = len(textimage_records)
# textimage_records = [r for r in textimage_records if r["orig_idx"] not in leaked_indices]
# after_count = len(textimage_records)

# logger.info("Dropped %d leaked rows (%d flagged by row_index)", before_count - after_count, len(leaked_indices))
# logger.info("Text/image records: %d -> %d", before_count, after_count)

# random.seed(RANDOM_SEED)
# random.shuffle(textimage_records)
# FULL_POOL_SIZE = len(textimage_records)
# logger.info("Full pool: %d records", FULL_POOL_SIZE)

# pkl_vars(leak_df, "vars/leak_df.pkl")
# pkl_vars(leaked_indices, "vars/leaked_indices.pkl")
# pkl_vars(textimage_records, "vars/textimage_records.pkl")
# pkl_vars(FULL_POOL_SIZE, "vars/FULL_POOL_SIZE.pkl")


## Section 3b — In-batch negative classification (hard/soft/drop)

Reinstated for these large sizes specifically: the rate of anchors with zero hard negatives per batch is
expected to rise as training size grows (a fixed batch of 32 becomes a smaller, more scattered slice of a
much larger, more diverse pool, so the chance any two batch members share season/region drops). This is
exactly the sizes where that trend needs to stay visible, not the sizes where it's safe to stop measuring
it — so unlike an earlier version of this notebook, hard/soft counting is NOT skipped here.

In [7]:
def classify_batch_negatives(batch):
    """batch: list of records with 'season', 'region_qid'.
    Returns {anchor_batch_idx: {"hard": [...other batch indices...], "soft": [...]}}.

    drop = same season AND same region (covers same-bulletin pairs by construction)
    hard = different bulletin AND (same season XOR same region)
    soft = different bulletin AND different season AND different region
    """
    B = len(batch)
    result = {}
    for i in range(B):
        hard, soft = [], []
        for j in range(B):
            if i == j:
                continue
            same_season = batch[i]["season"] == batch[j]["season"]
            same_region = batch[i]["region_qid"] == batch[j]["region_qid"]

            if same_season and same_region:
                continue  # dropped

            if same_season != same_region:  # XOR: exactly one matches
                hard.append(j)
            else:
                soft.append(j)
        result[i] = {"hard": hard, "soft": soft}
    return result

logger.info("classify_batch_negatives() ready.")

## Section 4 — Model builder function

Fresh DINOv3 + BGE-M3 (both last-6-layers-unfrozen) + fresh projection heads + fresh optimizer per size.

In [8]:
def build_models():
    dinov3_processor = AutoImageProcessor.from_pretrained(DINOV3_MODEL_NAME,cache_dir=HF_CACHE)
    dinov3_model = AutoModel.from_pretrained(DINOV3_MODEL_NAME,cache_dir=HF_CACHE).to(device)

    for p in dinov3_model.parameters():
        p.requires_grad = False
    dinov3_layers = dinov3_model.layer
    for layer in dinov3_layers[-N_UNFROZEN_LAYERS_DINOV3:]:
        for p in layer.parameters():
            p.requires_grad = True

    DINOV3_DIM = dinov3_model.config.hidden_size

    def encode_dinov3_images_batched(image_paths, chunk_size=IMAGE_ENCODE_CHUNK_SIZE):
        if len(image_paths) <= chunk_size:
            images = [Image.open(p).convert("RGB") for p in image_paths]
            inputs = dinov3_processor(images=images, return_tensors="pt").to(device)
            out = dinov3_model(**inputs)
            return out.last_hidden_state[:, 0, :]
        chunks = []
        for i in range(0, len(image_paths), chunk_size):
            batch_paths = image_paths[i:i + chunk_size]
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            inputs = dinov3_processor(images=images, return_tensors="pt").to(device)
            out = dinov3_model(**inputs)
            chunks.append(out.last_hidden_state[:, 0, :])
        return torch.cat(chunks, dim=0)

    bge_tokenizer = AutoTokenizer.from_pretrained(BGE_MODEL_NAME,cache_dir=HF_CACHE)
    bge_model = AutoModel.from_pretrained(BGE_MODEL_NAME,cache_dir=HF_CACHE).to(device)

    for p in bge_model.parameters():
        p.requires_grad = False
    bge_layers = bge_model.encoder.layer
    for layer in bge_layers[-N_UNFROZEN_LAYERS_BGE:]:
        for p in layer.parameters():
            p.requires_grad = True

    BGE_DIM = bge_model.config.hidden_size

    def encode_bge_text(texts):
        inputs = bge_tokenizer(texts, return_tensors="pt", padding=True, truncation=True,
                                 max_length=BGE_MAX_LENGTH).to(device)
        out = bge_model(**inputs)
        return out.last_hidden_state[:, 0, :]

    class ProjectionHead(nn.Module):
        def __init__(self, in_dim, out_dim=SHARED_DIM):
            super().__init__()
            self.proj = nn.Linear(in_dim, out_dim)

        def forward(self, x):
            return F.normalize(self.proj(x), p=2, dim=-1)

    head_text_bge = ProjectionHead(BGE_DIM).to(device)
    head_image_dinov3 = ProjectionHead(DINOV3_DIM).to(device)

    unfrozen_dinov3_params = [p for p in dinov3_model.parameters() if p.requires_grad]
    unfrozen_bge_params = [p for p in bge_model.parameters() if p.requires_grad]
    head_params = list(head_text_bge.parameters()) + list(head_image_dinov3.parameters())

    optimizer = torch.optim.AdamW([
        {"params": head_params, "lr": HEAD_LEARNING_RATE},
        {"params": unfrozen_dinov3_params, "lr": DINOV3_ENCODER_LR},
        {"params": unfrozen_bge_params, "lr": BGE_ENCODER_LR},
    ])

    return {
        "dinov3_model": dinov3_model, "encode_dinov3_images_batched": encode_dinov3_images_batched,
        "bge_model": bge_model, "encode_bge_text": encode_bge_text,
        "head_text_bge": head_text_bge, "head_image_dinov3": head_image_dinov3,
        "optimizer": optimizer,
    }

logger.info("build_models() ready.")

## Section 5 — Batch construction and the masked in-batch loss function (with hard/soft counting)

Counting reinstated per the observation above -- classifies every batch's pairs into hard/soft/drop and
accumulates counts, same as the original completed sizes, so the zero-hard-anchor trend stays visible at
these largest sizes too.

In [9]:
def make_batches(records, batch_size=BATCH_SIZE, shuffle=True):
    idx = np.arange(len(records))
    if shuffle:
        np.random.shuffle(idx)
    for start in range(0, len(idx), batch_size):
        batch_idx = idx[start:start + batch_size]
        yield [records[i] for i in batch_idx]


def compute_imagetext_loss_inbatch_mined(batch, models, batch_stats):
    """In-batch mined negatives, WITH hard/soft classification and counting (reinstated for these large
    sizes -- see the markdown note above). batch_stats is a dict this function ACCUMULATES into
    (hard_count, soft_count, dropped_count, zero_hard_anchors, zero_total_anchors, n_batches, n_anchors)."""
    B = len(batch)

    texts = [r["text"] for r in batch]
    image_paths = [r["image_path"] for r in batch]

    text_emb_raw = models["encode_bge_text"](texts)
    image_embs_raw = models["encode_dinov3_images_batched"](image_paths)

    text_emb = models["head_text_bge"](text_emb_raw)
    image_emb = models["head_image_dinov3"](image_embs_raw)

    sim = text_emb @ image_emb.T

    classification = classify_batch_negatives(batch)

    mask = torch.zeros((B, B), dtype=torch.bool, device=device)
    for i in range(B):
        info = classification[i]
        hard_n, soft_n = len(info["hard"]), len(info["soft"])
        batch_stats["hard_count"] += hard_n
        batch_stats["soft_count"] += soft_n
        batch_stats["dropped_count"] += (B - 1 - hard_n - soft_n)
        if hard_n == 0:
            batch_stats["zero_hard_anchors"] += 1
        if hard_n + soft_n == 0:
            batch_stats["zero_total_anchors"] += 1
        keep = set(info["hard"]) | set(info["soft"]) | {i}
        for j in range(B):
            if j not in keep:
                mask[i, j] = True
    batch_stats["n_batches"] += 1
    batch_stats["n_anchors"] += B

    logits = sim / TEMPERATURE
    neg_inf = torch.finfo(logits.dtype).min
    logits_masked = logits.masked_fill(mask, neg_inf)

    targets = torch.arange(B, device=device)
    loss_t2i = F.cross_entropy(logits_masked, targets)
    loss_i2t = F.cross_entropy(logits_masked.T, targets)
    return (loss_t2i + loss_i2t) / 2

logger.info("Batch construction and in-batch-mined loss function (with counting) ready.")

## Section 6 — Load GT set

In [10]:
gt_textimage_records = reload_vars("vars/gt_textimage_records.pkl")
raw_gt_textimage = reload_vars("vars/raw_gt_textimage.pkl")

In [11]:
# raw_gt_textimage = load_jsonl(GT_TEXTIMAGE_PATH)
# logger.info("GT text/image rows: %d", len(raw_gt_textimage))

# gt_textimage_records = []
# for i, r in enumerate(raw_gt_textimage):
#     text = r.get("text", "").strip()
#     filename = r.get("image", {}).get("filename")
#     if not text or not filename:
#         continue
#     img_path = os.path.join(GT_IMAGE_DIR, filename)
#     if not os.path.exists(img_path):
#         continue
#     gt_textimage_records.append({"gt_idx": i, "text": text, "image_path": img_path})
# logger.info("Usable GT text/image records: %d / %d", len(gt_textimage_records), len(raw_gt_textimage))
# assert len(gt_textimage_records) > 0

# pkl_vars(gt_textimage_records, "vars/gt_textimage_records.pkl")
# pkl_vars(raw_gt_textimage, "vars/raw_gt_textimage.pkl")

## Section 7 — GT evaluation function

In [11]:
def compute_retrieval_metrics(sim_matrix, k_values=K_VALUES):
    N = sim_matrix.shape[0]
    true_scores = sim_matrix.diag().unsqueeze(1)
    ranks = (sim_matrix >= true_scores).sum(dim=1)
    results = {}
    for k in k_values:
        results[f"R@{k}"] = (ranks <= k).float().mean().item()
    results["MRR"] = (1.0 / ranks.float()).mean().item()
    results["N"] = N
    return results


def evaluate_on_gt(models):
    models["head_text_bge"].eval()
    models["head_image_dinov3"].eval()
    models["dinov3_model"].eval()
    models["bge_model"].eval()

    with torch.no_grad():
        gt_image_emb = models["head_image_dinov3"](
            models["encode_dinov3_images_batched"]([r["image_path"] for r in gt_textimage_records])
        )
        gt_text_emb = models["head_text_bge"](
            models["encode_bge_text"]([r["text"] for r in gt_textimage_records])
        )

    sim_imagetext = gt_image_emb @ gt_text_emb.T
    results_text_to_image = compute_retrieval_metrics(sim_imagetext.T)
    results_image_to_text = compute_retrieval_metrics(sim_imagetext)

    summary = pd.DataFrame([
        {"direction": "text -> image", **results_text_to_image},
        {"direction": "image -> text", **results_image_to_text},
    ])[["direction", "N", "R@1", "R@3", "R@5", "R@10", "MRR"]]

    models["head_text_bge"].train()
    models["head_image_dinov3"].train()
    models["dinov3_model"].train()
    models["bge_model"].train()
    return summary

logger.info("evaluate_on_gt() ready.")

## Section 8 — Sanity check (RUN THIS BEFORE STARTING)

In [12]:
_sanity_models = build_models()
_sanity_batch = textimage_records[:BATCH_SIZE]
_sanity_stats = {"hard_count": 0, "soft_count": 0, "dropped_count": 0, "zero_hard_anchors": 0,
                  "zero_total_anchors": 0, "n_batches": 0, "n_anchors": 0}
loss_check = compute_imagetext_loss_inbatch_mined(_sanity_batch, _sanity_models, _sanity_stats)

logger.info("Sanity loss: %.4f (naive upper bound ln(%d) = %.4f)", loss_check.item(), BATCH_SIZE, math.log(BATCH_SIZE))
logger.info("Sanity batch stats: avg hard/anchor=%.1f, avg soft/anchor=%.1f, zero-hard anchors=%d/%d",
            _sanity_stats["hard_count"]/BATCH_SIZE, _sanity_stats["soft_count"]/BATCH_SIZE,
            _sanity_stats["zero_hard_anchors"], BATCH_SIZE)
logger.info("If NaN, stop and debug before running the full sweep.")

loss_check.backward()
grad_dinov3 = any(p.grad is not None and p.grad.abs().sum().item() > 0 for p in _sanity_models["dinov3_model"].parameters() if p.requires_grad)
grad_bge = any(p.grad is not None and p.grad.abs().sum().item() > 0 for p in _sanity_models["bge_model"].parameters() if p.requires_grad)
logger.info("Gradients reaching unfrozen DINOv3 layers: %s", grad_dinov3)
logger.info("Gradients reaching unfrozen BGE-M3 layers: %s", grad_bge)

del _sanity_models, loss_check, _sanity_batch, _sanity_stats
import gc
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
logger.info("Allocated after cleanup: %.2f GB", torch.cuda.memory_allocated() / 1e9)

## Section 9 — Train the two remaining sizes

Epoch counts (8 for N=20000, 5 for N=33208) match the tapered schedule confirmed earlier in this project,
not a flat count. Skip-if-already-done, atomic checkpoint writes, widened logging/checkpoint intervals.

In [ ]:
#EPOCHS_BY_SIZE = {33208: 20, 20000: 20}

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

sweep_summary_rows = []

for N in REMAINING_SIZES:
    size_dir = os.path.join(OUTPUT_DIR, f"size_{N}")
    os.makedirs(size_dir, exist_ok=True)
    gt_results_path = os.path.join(size_dir, "gt_results.csv")
    ckpt_path = os.path.join(size_dir, "latest_checkpoint.pt")

    if os.path.exists(gt_results_path):
        logger.info("=== N=%d already completed, skipping ===", N)
        sweep_summary_rows.append(pd.read_csv(gt_results_path))
        continue

    logger.info("=" * 70)
    logger.info("=== N=%d ===", N)
    logger.info("=" * 70)

    size_rng = random.Random(RANDOM_SEED + N)
    size_records = list(textimage_records) if N >= FULL_POOL_SIZE else size_rng.sample(textimage_records, N)
    logger.info("Records for this size: %d", len(size_records))

    num_epochs = 20 #EPOCHS_BY_SIZE[N]
    models = build_models()

    start_epoch, global_step = 0, 0
    history = {"epoch": [], "imagetext_loss": [], "avg_hard_per_anchor": [], "avg_soft_per_anchor": [],
               "zero_hard_anchors": [], "zero_total_anchors": []}

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=device)
        models["head_text_bge"].load_state_dict(ckpt["head_text_bge"])
        models["head_image_dinov3"].load_state_dict(ckpt["head_image_dinov3"])
        models["dinov3_model"].load_state_dict(ckpt["dinov3_model"])
        models["bge_model"].load_state_dict(ckpt["bge_model"])
        models["optimizer"].load_state_dict(ckpt["optimizer"])
        start_epoch = ckpt["epoch"]
        global_step = ckpt["global_step"]
        history = ckpt.get("history", history)
        logger.info("Resumed N=%d from checkpoint: epoch %d, global_step %d", N, start_epoch, global_step)
    else:
        logger.info("No checkpoint found for N=%d, starting fresh.", N)

    def save_checkpoint(epoch, global_step, history):
        tmp_path = ckpt_path + ".tmp"
        torch.save({
            "head_text_bge": models["head_text_bge"].state_dict(),
            "head_image_dinov3": models["head_image_dinov3"].state_dict(),
            "dinov3_model": models["dinov3_model"].state_dict(),
            "bge_model": models["bge_model"].state_dict(),
            "optimizer": models["optimizer"].state_dict(),
            "epoch": epoch, "global_step": global_step, "history": history,
        }, tmp_path)
        os.replace(tmp_path, ckpt_path)

    training_start_time = time.time()
    stop_early = False

    try:
        for epoch in range(start_epoch, num_epochs):
            epoch_losses = []
            epoch_stats = {"hard_count": 0, "soft_count": 0, "dropped_count": 0, "zero_hard_anchors": 0,
                            "zero_total_anchors": 0, "n_batches": 0, "n_anchors": 0}
            batches = list(make_batches(size_records))
            n_steps = len(batches)

            for step in range(n_steps):
                step_start = time.time()
                models["optimizer"].zero_grad()

                loss = compute_imagetext_loss_inbatch_mined(batches[step], models, epoch_stats)
                loss.backward()
                models["optimizer"].step()
                global_step += 1
                if device == "cuda":
                    torch.cuda.empty_cache()
                epoch_losses.append(loss.item())

                elapsed_hours = (time.time() - training_start_time) / 3600
                if step % LOG_EVERY_N_STEPS == 0:
                    step_time = time.time() - step_start
                    logger.info("[N=%d] Epoch %d/%d, step %d/%d, loss=%.4f, step_time=%.2fs, elapsed=%.2fh",
                                N, epoch + 1, num_epochs, step, n_steps, loss.item(), step_time, elapsed_hours)

                if global_step % CHECKPOINT_EVERY_N_STEPS == 0:
                    save_checkpoint(epoch, global_step, history)

                if elapsed_hours > MAX_RUNTIME_HOURS_PER_SIZE:
                    logger.warning("Approaching runtime limit for N=%d. Saving and stopping.", N)
                    save_checkpoint(epoch, global_step, history)
                    stop_early = True
                    break

            if stop_early:
                break

            mean_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
            avg_hard = epoch_stats["hard_count"] / max(epoch_stats["n_anchors"], 1)
            avg_soft = epoch_stats["soft_count"] / max(epoch_stats["n_anchors"], 1)
            history["epoch"].append(epoch + 1)
            history["imagetext_loss"].append(mean_loss)
            history["avg_hard_per_anchor"].append(avg_hard)
            history["avg_soft_per_anchor"].append(avg_soft)
            history["zero_hard_anchors"].append(epoch_stats["zero_hard_anchors"])
            history["zero_total_anchors"].append(epoch_stats["zero_total_anchors"])
            logger.info("[N=%d] Epoch %d done: loss=%.4f, avg_hard/anchor=%.1f, avg_soft/anchor=%.1f, "
                        "zero_hard_anchors=%d, zero_total_anchors=%d",
                        N, epoch + 1, mean_loss, avg_hard, avg_soft,
                        epoch_stats["zero_hard_anchors"], epoch_stats["zero_total_anchors"])
            gt_summary = evaluate_on_gt(models)
            gt_summary.insert(0, "train_size", N)
            logger.info("GT results for N=%d:\n%s", N, gt_summary.to_string(index=False))
            save_checkpoint(epoch + 1, global_step, history)

        pd.DataFrame(history).to_csv(os.path.join(size_dir, "training_history.csv"), index=False)
        torch.save(models["head_text_bge"].state_dict(), os.path.join(size_dir, "head_text_bge.pt"))
        torch.save(models["head_image_dinov3"].state_dict(), os.path.join(size_dir, "head_image_dinov3.pt"))
        torch.save(models["dinov3_model"].state_dict(), os.path.join(size_dir, "dinov3_model_unfrozen.pt"))
        torch.save(models["bge_model"].state_dict(), os.path.join(size_dir, "bge_model_unfrozen.pt"))
        logger.info("Saved outputs for N=%d to %s", N, size_dir)

        gt_summary = evaluate_on_gt(models)
        gt_summary.insert(0, "train_size", N)
        gt_summary.to_csv(gt_results_path, index=False)
        logger.info("GT results for N=%d:\n%s", N, gt_summary.to_string(index=False))
        sweep_summary_rows.append(gt_summary)

    except torch.cuda.OutOfMemoryError:
        logger.error("OOM on N=%d. Skipping to next size.", N)

    finally:
        try:
            del models
        except NameError:
            pass
        import gc
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

logger.info("=" * 70)
logger.info("Remaining sizes complete.")
logger.info("=" * 70)

## Section 10 — Full combined summary (all 8 sizes, since this shares the completed run's directory)

In [ ]:
ALL_SIZES = [100, 500, 1000, 2500, 5000, 10000, 20000, 33208]

rows = []
for N in ALL_SIZES:
    path = os.path.join(OUTPUT_DIR, f"size_{N}", "gt_results.csv")
    if os.path.exists(path):
        rows.append(pd.read_csv(path))
    else:
        logger.warning("N=%d: no gt_results.csv found, skipping", N)

summary_all_sizes = pd.concat(rows, ignore_index=True)
summary_all_sizes.to_csv(os.path.join(OUTPUT_DIR, "summary_all_sizes.csv"), index=False)
logger.info("=== Full summary across all sizes ===\n%s", summary_all_sizes.to_string(index=False))
display(summary_all_sizes)